In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/konicarokeya/mesh-complete/retrieval_corpus_MESH_COMPLETE.parquet
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ac/MRREL.RRF.ac
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ae/MRREL.RRF.ae
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ad/MRREL.RRF.ad
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRCONSO.RRF.aa/MRCONSO.RRF.aa
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ab/MRREL.RRF.ab
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRSTY.RRF/MRSTY.RRF
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.aa/MRREL.RRF.aa
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRCONSO.RRF.ab/MRCONSO.RRF.ab
/kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRCONSO.RRF.ac/MRCONSO.RRF.ac


In [4]:
from pathlib import Path

# Check what the input path actually is
base = Path('/kaggle/input')
print('All input datasets:')
for d in sorted(base.iterdir()):
    print(f'  {d.name}')

print()

# Find the umls dataset
for d in sorted(base.iterdir()):
    if 'umls' in d.name.lower():
        print(f'Found UMLS dataset: {d}')
        print('Files inside:')
        for f in sorted(d.rglob('*')):
            if f.is_file():
                print(f'  {f.relative_to(base)}  ({f.stat().st_size/1024**2:.0f} MB)')

All input datasets:
  datasets



In [5]:
from pathlib import Path

base = Path('/kaggle/input/datasets')
print('Contents of datasets folder:')
for d in sorted(base.iterdir()):
    print(f'  {d.name}')

print()
print('All files (searching deep):')
for f in sorted(base.rglob('*')):
    if f.is_file():
        print(f'  {f}  ({f.stat().st_size/1024**2:.0f} MB)')

Contents of datasets folder:
  konicarokeya

All files (searching deep):
  /kaggle/input/datasets/konicarokeya/mesh-complete/retrieval_corpus_MESH_COMPLETE.parquet  (779 MB)
  /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRCONSO.RRF.aa/MRCONSO.RRF.aa  (1024 MB)
  /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRCONSO.RRF.ab/MRCONSO.RRF.ab  (1024 MB)
  /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRCONSO.RRF.ac/MRCONSO.RRF.ac  (97 MB)
  /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.aa/MRREL.RRF.aa  (1024 MB)
  /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ab/MRREL.RRF.ab  (1024 MB)
  /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ac/MRREL.RRF.ac  (1024 MB)
  /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ad/MRREL.RRF.ad  (1024 MB)
  /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRREL.RRF.ae/MRREL.RRF.ae  (89 MB)
  /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRSTY.RRF/MRSTY.RR

In [9]:
from pathlib import Path
import shutil

UMLS_BASE = Path('/kaggle/input/datasets/konicarokeya/umls-2025ab-parts')
UMLS_OUT  = Path('/kaggle/working/umls')
UMLS_OUT.mkdir(exist_ok=True)

def concat_parts(part_names, output_name):
    out_path = UMLS_OUT / output_name
    if out_path.exists():
        print(f'{output_name} already exists ({out_path.stat().st_size/1024**2:.0f} MB) — skipping')
        return out_path
    print(f'Building {output_name} from {len(part_names)} parts...')
    with open(out_path, 'wb') as out_f:
        for name in sorted(part_names):
            p = UMLS_BASE / name / name  # each file is inside its own subfolder
            print(f'  + {name}  ({p.stat().st_size/1024**2:.0f} MB)')
            with open(p, 'rb') as in_f:
                shutil.copyfileobj(in_f, out_f)
    print(f'  done -> {out_path.stat().st_size/1024**2:.0f} MB total\n')
    return out_path

# MRCONSO — 3 parts
MRCONSO_PATH = concat_parts(
    ['MRCONSO.RRF.aa', 'MRCONSO.RRF.ab', 'MRCONSO.RRF.ac'],
    'MRCONSO.RRF'
)

# MRREL — 5 parts
MRREL_PATH = concat_parts(
    ['MRREL.RRF.aa', 'MRREL.RRF.ab', 'MRREL.RRF.ac',
     'MRREL.RRF.ad', 'MRREL.RRF.ae'],
    'MRREL.RRF'
)

# MRSTY — already complete, just reference it directly
MRSTY_PATH = UMLS_BASE / 'MRSTY.RRF' / 'MRSTY.RRF'

# Corpus parquet
CORPUS_PATH = Path('/kaggle/input/datasets/konicarokeya/mesh-complete/retrieval_corpus_MESH_COMPLETE.parquet')

print('='*50)
print('Paths ready:')
print(f'  MRCONSO : {MRCONSO_PATH}')
print(f'  MRREL   : {MRREL_PATH}')
print(f'  MRSTY   : {MRSTY_PATH}')
print(f'  CORPUS  : {CORPUS_PATH}')
print(f'  MRSTY exists : {MRSTY_PATH.exists()}')
print(f'  CORPUS exists: {CORPUS_PATH.exists()}')

Building MRCONSO.RRF from 3 parts...
  + MRCONSO.RRF.aa  (1024 MB)
  + MRCONSO.RRF.ab  (1024 MB)
  + MRCONSO.RRF.ac  (97 MB)
  done -> 2145 MB total

Building MRREL.RRF from 5 parts...
  + MRREL.RRF.aa  (1024 MB)
  + MRREL.RRF.ab  (1024 MB)
  + MRREL.RRF.ac  (1024 MB)
  + MRREL.RRF.ad  (1024 MB)
  + MRREL.RRF.ae  (89 MB)
  done -> 4185 MB total

Paths ready:
  MRCONSO : /kaggle/working/umls/MRCONSO.RRF
  MRREL   : /kaggle/working/umls/MRREL.RRF
  MRSTY   : /kaggle/input/datasets/konicarokeya/umls-2025ab-parts/MRSTY.RRF/MRSTY.RRF
  CORPUS  : /kaggle/input/datasets/konicarokeya/mesh-complete/retrieval_corpus_MESH_COMPLETE.parquet
  MRSTY exists : True
  CORPUS exists: True


In [7]:
import gc
import math
import pickle
import csv
from pathlib import Path
from collections import defaultdict
from itertools import combinations
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import networkx as nx

print('Imports OK')

Imports OK


In [8]:
import os
from pathlib import Path

# Delete concatenated RRF files (they take 6+ GB)
umls_out = Path('/kaggle/working/umls')
for f in umls_out.iterdir():
    print(f'Deleting {f.name} ({f.stat().st_size/1024**2:.0f} MB)...')
    f.unlink()
umls_out.rmdir()

# Delete old hkg.pkl
hkg = Path('/kaggle/working/hkg.pkl')
if hkg.exists():
    print(f'Deleting hkg.pkl ({hkg.stat().st_size/1024**2:.0f} MB)...')
    hkg.unlink()

print('Done. All cache cleared.')

# Check remaining space
import shutil
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'Free space: {free/1024**3:.1f} GB')

Deleting MRCONSO.RRF (2145 MB)...
Deleting MRREL.RRF (4185 MB)...
Done. All cache cleared.
Free space: 19.5 GB


In [19]:
# ── Paths (set by Cell 0) ─────────────────────────────────────
# MRCONSO_PATH, MRREL_PATH, MRSTY_PATH, CORPUS_PATH already set

HKG_OUT = Path('/kaggle/working/hkg.pkl')

# ── UMLS filter ───────────────────────────────────────────────
TARGET_SABS = {'MSH', 'SNOMEDCT_US'}

# Only keep these 5 semantic groups (TUI prefixes)
# T047=Disease, T121=Drug, T023=Anatomy, T039=Physiology, T061=Procedure
ALLOWED_TUIS = {
    # Disorders
    'T047','T048','T049','T050','T191',
    # Chemicals & Drugs
    'T121','T109','T195','T200','T116',
    # Anatomy
    'T023','T024','T025','T026','T029',
    # Physiology
    'T039','T040','T041','T042','T043',
    # Procedures
    'T061','T058','T059','T060','T065',
    # Population & Demographics — female, male
    'T016',  # Human
    'T096',  # Group
    'T098',  # Population Group
    'T099',  # Family Group
    'T032',  # Organism Attribute — female/male
    # Age groups — adult, child, infant, aged
    'T100',  # Age Group
    'T079',  # Temporal Concept — time factors, age factors
    # Animals — rats, mice, rabbits, dogs
    'T008',  # Animal
    'T015',  # Mammal
    # Research concepts
    'T062',  # Research Activity
    'T081',  # Quantitative Concept
    'T077',  # Intellectual Product — epidemiology
    # Findings & Results
    'T033',  # Finding — postoperative complications, treatment outcome
    'T034',  # Laboratory or Test Result
    'T123',  # Biologically Active Substance — biomarkers
    # Molecular biology
    'T086',  # Nucleotide Sequence — base sequence
    'T114',  # Nucleic Acid — DNA, RNA
    'T044',  # Molecular Function — gene expression regulation
}

# Co-occurrence minimum
MIN_COOCCUR = 2

print('Config OK')
print(f'  Target SABs : {TARGET_SABS}')
print(f'  Allowed TUIs: {len(ALLOWED_TUIS)}')

Config OK
  Target SABs : {'MSH', 'SNOMEDCT_US'}
  Allowed TUIs: 43


In [11]:
print('Loading corpus...')
df = pd.read_parquet(CORPUS_PATH)
print(f'Rows: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f'Sources: {df["source"].value_counts().to_dict()}')

def normalize_mesh(m):
    if m is None: return []
    if isinstance(m, np.ndarray): m = m.tolist()
    if not isinstance(m, list): return []
    out = []
    for x in m:
        if isinstance(x, dict):
            term   = x.get('term','')
            meshid = x.get('mesh_id','')
            if term and str(term).strip():
                out.append({
                    'term'   : str(term).strip().lower(),
                    'mesh_id': str(meshid).strip() if meshid else ''
                })
    return out

df['meshes_norm'] = df['meshes'].apply(normalize_mesh)

# Build term→chunks and meshid→chunks
term_to_chunks  = defaultdict(list)
meshid_to_chunks = defaultdict(list)

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Building chunk index')):
    for entry in mesh_list:
        t = entry['term']
        m = entry['mesh_id']
        if t:
            term_to_chunks[t].append(idx)
        if m and m.startswith('D'):
            meshid_to_chunks[m].append(idx)

corpus_texts = df['text'].tolist()

print(f'\nUnique terms     : {len(term_to_chunks):,}')
print(f'Unique mesh_ids  : {len(meshid_to_chunks):,}')
print(f'Corpus size      : {len(corpus_texts):,}')
gc.collect()

Loading corpus...
Rows: 2,089,296
Columns: ['chunk_id', 'original_corpus_id', 'chunk_index', 'total_chunks', 'text', 'num_chars', 'is_chunked', 'pmid', 'source', 'context_index', 'total_contexts_in_paper', 'question', 'meshes', 'section_label', 'final_decision', 'year', 'has_long_answer']
Sources: {'medrag_pubmed': 1000000, 'pqaa': 649412, 'medrag_wikipedia': 239348, 'pqau': 200536}


Building chunk index:   0%|          | 0/2089296 [00:00<?, ?it/s]


Unique terms     : 26,190
Unique mesh_ids  : 26,177
Corpus size      : 2,089,296


20

In [20]:
print('Loading MRSTY — semantic type filter...')

allowed_cuis = set()
cui_to_tui   = {}

with open(MRSTY_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRSTY'):
        row = line.rstrip('\n').split('|')
        if len(row) < 2: continue
        cui = row[0]
        tui = row[1]
        cui_to_tui[cui] = tui
        if tui in ALLOWED_TUIS:
            allowed_cuis.add(cui)

print(f'Total CUIs in MRSTY  : {len(cui_to_tui):,}')
print(f'Allowed CUIs (filter): {len(allowed_cuis):,}')
gc.collect()

Loading MRSTY — semantic type filter...


MRSTY: 0it [00:00, ?it/s]

Total CUIs in MRSTY  : 3,488,973
Allowed CUIs (filter): 2,032,387


18

In [21]:
print('Loading MRCONSO — concept nodes...')

G             = nx.DiGraph()
label_to_cui  = {}   # term string → CUI
meshid_to_cui = {}   # D-number   → CUI

with open(MRCONSO_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRCONSO'):
        row = line.rstrip('\n').split('|')
        if len(row) < 15: continue
        cui    = row[0]
        lang   = row[1]
        ispref = row[6]
        sab    = row[11]
        tty    = row[12]
        code   = row[13]
        name   = row[14]

        if lang != 'ENG': continue
        if sab not in TARGET_SABS: continue
        if ispref != 'Y': continue
        if cui not in allowed_cuis: continue

        label = name.lower().strip()

        if cui not in G:
            G.add_node(cui, label=label, sab=sab, chunk_idxs=[])

        # Map term string → CUI
        if label:
            label_to_cui[label] = cui

        # Map MeSH D-number → CUI
        if sab == 'MSH' and code.startswith('D'):
            meshid_to_cui[code] = cui

print(f'Nodes loaded : {G.number_of_nodes():,}')
print(f'label_to_cui : {len(label_to_cui):,}')
print(f'meshid_to_cui: {len(meshid_to_cui):,}')
gc.collect()

Loading MRCONSO — concept nodes...


MRCONSO: 0it [00:00, ?it/s]

Nodes loaded : 640,565
label_to_cui : 1,757,477
meshid_to_cui: 20,682


18

In [22]:
print('Loading MRREL — edges...')

VALID_RELS = {'PAR','CHD','RB','RN','RO','SY'}
edge_count = 0

with open(MRREL_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRREL'):
        row = line.rstrip('\n').split('|')
        if len(row) < 8: continue
        cui1 = row[0]
        rel  = row[3]
        cui2 = row[4]
        rela = row[7]

        if rel not in VALID_RELS: continue
        if cui1 not in G: continue
        if cui2 not in G: continue

        G.add_edge(cui1, cui2, rel=rel, rela=rela or rel)
        edge_count += 1

print(f'UMLS edges added: {edge_count:,}')
print(f'Total edges now : {G.number_of_edges():,}')
gc.collect()

Loading MRREL — edges...


MRREL: 0it [00:00, ?it/s]

UMLS edges added: 4,256,508
Total edges now : 1,201,014


18

In [23]:
print('Connecting corpus chunks to graph nodes...')

connected_via_meshid = 0
connected_via_string = 0
no_node_found        = 0

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Connecting')):
    for entry in mesh_list:
        term   = entry['term']
        meshid = entry['mesh_id']

        # Method 1: D-number → CUI (reliable, pqaa/pqau/pubmed)
        if meshid and meshid in meshid_to_cui:
            cui = meshid_to_cui[meshid]
            if cui in G:
                G.nodes[cui]['chunk_idxs'].append(idx)
                connected_via_meshid += 1
                continue

        # Method 2: term string → CUI (fallback, wikipedia)
        if term and term in label_to_cui:
            cui = label_to_cui[term]
            if cui in G:
                G.nodes[cui]['chunk_idxs'].append(idx)
                connected_via_string += 1
                continue

        no_node_found += 1

# Deduplicate chunk_idxs
for node in G.nodes():
    G.nodes[node]['chunk_idxs'] = list(set(G.nodes[node]['chunk_idxs']))

reachable = sum(1 for n in G.nodes() if G.nodes[n]['chunk_idxs'])
print(f'\nConnected via mesh_id : {connected_via_meshid:,}')
print(f'Connected via string  : {connected_via_string:,}')
print(f'No node found         : {no_node_found:,}')
print(f'Nodes with chunks     : {reachable:,} / {G.number_of_nodes():,}')
gc.collect()

Connecting corpus chunks to graph nodes...


Connecting:   0%|          | 0/2089296 [00:00<?, ?it/s]


Connected via mesh_id : 18,974,703
Connected via string  : 170,058
No node found         : 3,963,452
Nodes with chunks     : 18,548 / 640,565


21

In [24]:
from collections import Counter

missing_terms   = Counter()
missing_meshids = Counter()

for idx, mesh_list in enumerate(df['meshes_norm']):
    for entry in mesh_list:
        term   = entry['term']
        meshid = entry['mesh_id']
        if meshid and meshid in meshid_to_cui:
            continue
        if term and term in label_to_cui:
            continue
        if meshid and meshid.startswith('D'):
            missing_meshids[meshid] += 1
        if term:
            missing_terms[term] += 1

print('Top 20 missing TERMS:')
for term, count in missing_terms.most_common(20):
    print(f'  {count:>8,}  {term}')

print(f'\nTotal unique missing terms   : {len(missing_terms):,}')
print(f'Total unique missing mesh_ids: {len(missing_meshids):,}')

Top 20 missing TERMS:
   556,656  middle aged
    43,897  kinetics
    39,511  molecular sequence data
    39,288  postoperative complications
    38,233  biomarkers
    37,911  treatment outcome
    36,753  epidemiology
    31,150  amino acid sequence
    26,933  gene expression regulation
    25,859  mutation
    23,459  body mass index
    22,062  united states
    21,769  sex factors
    20,312  recurrence
    19,312  gene expression
    19,078  inflammation
    18,539  escherichia coli
    18,530  obesity
    17,892  cloning, molecular
    17,137  transcription, genetic

Total unique missing terms   : 7,653
Total unique missing mesh_ids: 7,632


In [16]:
# Find exactly which terms are causing "no node found"
from collections import Counter

missing_terms   = Counter()
missing_meshids = Counter()

for idx, mesh_list in enumerate(df['meshes_norm']):
    for entry in mesh_list:
        term   = entry['term']
        meshid = entry['mesh_id']

        # Check mesh_id
        if meshid and meshid in meshid_to_cui:
            continue
        # Check string
        if term and term in label_to_cui:
            continue

        # This entry had no node found
        if meshid and meshid.startswith('D'):
            missing_meshids[meshid] += 1
        if term:
            missing_terms[term] += 1

print('Top 30 missing TERMS:')
for term, count in missing_terms.most_common(30):
    print(f'  {count:>8,}  {term}')

print()
print('Top 20 missing MESH IDs:')
for mid, count in missing_meshids.most_common(20):
    print(f'  {count:>8,}  {mid}')

print()
print(f'Total unique missing terms   : {len(missing_terms):,}')
print(f'Total unique missing mesh_ids: {len(missing_meshids):,}')

Top 30 missing TERMS:
   933,816  female
   931,964  male
   596,480  adult
   556,656  middle aged
   500,817  animals
   146,988  rats
   142,443  mice
   139,381  child
   127,338  aged, 80 and over
   100,209  time factors
    85,340  child, preschool
    65,122  infant
    51,001  age factors
    48,947  infant, newborn
    45,267  rats, inbred strains
    43,897  kinetics
    40,136  base sequence
    39,511  molecular sequence data
    39,288  postoperative complications
    38,233  biomarkers
    37,911  treatment outcome
    36,753  epidemiology
    36,323  rna, messenger
    31,150  amino acid sequence
    30,850  mice, inbred c57bl
    29,454  rabbits
    29,146  dogs
    28,927  young adult
    28,639  dna
    26,933  gene expression regulation

Top 20 missing MESH IDs:
   933,816  D005260
   931,964  D008297
   596,480  D000328
   556,656  D008875
   500,817  D000818
   146,988  D051381
   142,443  D051379
   139,381  D002648
   127,338  D000369
   100,209  D013997
    85,

In [15]:
print('Building co-occurrence edges...')

cooccur = defaultdict(int)

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Co-occurrence')):
    terms = list(set(
        meshid_to_cui.get(e['mesh_id'], label_to_cui.get(e['term']))
        for e in mesh_list
        if meshid_to_cui.get(e['mesh_id']) or label_to_cui.get(e['term'])
    ))
    terms = [t for t in terms if t and t in G]
    for a, b in combinations(terms, 2):
        cooccur[(a, b)] += 1

co_edges = 0
for (a, b), w in cooccur.items():
    if w >= MIN_COOCCUR:
        G.add_edge(a, b, rel='CO_OCCUR', weight=w)
        G.add_edge(b, a, rel='CO_OCCUR', weight=w)
        co_edges += 1

print(f'Co-occurrence edges added: {co_edges:,}')
print(f'Total edges now          : {G.number_of_edges():,}')
del cooccur
gc.collect()

Building co-occurrence edges...


Co-occurrence:   0%|          | 0/2089296 [00:00<?, ?it/s]

Co-occurrence edges added: 2,769,076
Total edges now          : 6,142,424


20

In [16]:
print('Saving HKG...')

payload = {
    'graph'          : G,
    'term_to_chunks' : dict(term_to_chunks),
    'meshid_to_chunks': dict(meshid_to_chunks),
    'label_to_cui'   : label_to_cui,
    'meshid_to_cui'  : meshid_to_cui,
}

with open(HKG_OUT, 'wb') as f:
    pickle.dump(payload, f, protocol=4)

size_mb = HKG_OUT.stat().st_size / 1024**2
print(f'\nSaved: {HKG_OUT}  ({size_mb:.0f} MB)')
print(f'Nodes : {G.number_of_nodes():,}')
print(f'Edges : {G.number_of_edges():,}')
print('\nKG builder complete. Upload hkg.pkl as a Kaggle dataset.')

Saving HKG...

Saved: /kaggle/working/hkg.pkl  (601 MB)
Nodes : 567,230
Edges : 6,142,424

KG builder complete. Upload hkg.pkl as a Kaggle dataset.
